# AI ANALYST LAB

![](../_img/Ghost_TheSyntheticBanner.png)

### A Hands-on Course on AI for Data Analysts
## Review 01: Augmenting Data Analysis with LLMs

Feedback should be sent to [goran.milovanovic@datakolektiv.com](mailto:goran.milovanovic@datakolektiv.com).

This is a **review notebook**. It does not introduce new statistics. Instead, it stops to consolidate one single skill that has been growing quietly across Sessions 01–04: **using a Large Language Model — Anthropic's Claude — as a working tool inside a data-analysis workflow.** We will build the whole idea again from the ground up, assuming you remember nothing, and we will end with you confidently writing your own Claude API calls.

***
### What we will do today

Across the first four sessions you have, perhaps without quite noticing it, called Claude from a notebook four times: a plain-text stakeholder note (Session 01), a short interpretation paragraph (Session 02), a schema-checked A/B-test verdict (Session 03), and a structured drivers plan (Session 04). Each time, the API call was a small box at the end of the session, and we did not slow down to explain *what was actually happening inside that box*.

This review is that slow-down. We are going to take everything apart and rebuild it, piece by piece, **assuming you have never written an API call, never seen a Python dictionary, and never heard the word "JSON" in your life.** By the end you will understand every line of every Claude call you have made, and you will be able to write new ones on your own.

We will move in a deliberate order — each idea is a brick the next idea stands on:

| Section | What happens |
|---|---|
| R1.1 | **Why** put an LLM in a data-analysis workflow at all? |
| R1.2 | The one rule that keeps us honest: **"Python computes, the LLM interprets."** |
| R1.3 | Python building blocks: **lists, dictionaries, tuples** |
| R1.4 | **JSON** — the universal text format for structured data |
| R1.5 | **What an API is**, explained with no jargon |
| R1.6 | Setting up the **Anthropic Python SDK** and your API key |
| R1.7 | The **anatomy** of `client.messages.create(...)` |
| R1.8 | **Reading the response** object Claude sends back |
| R1.9 | **Choosing a model** — Haiku, Sonnet, Opus, and when to use each |
| R1.10 | **Writing good prompts** (with Anthropic's official prompt-engineering material) |
| R1.11 | **Pattern 1 — plain text**: ask, receive a paragraph |
| R1.12 | **Pattern 2 — free-form JSON**: ask for structured text, validate it yourself |
| R1.13 | Python **classes and objects** — the object model, from scratch |
| R1.14 | **Pydantic** — turning a class into a data contract |
| R1.15 | **Pattern 3 — tool use + Pydantic**: schema enforced at generation time |
| R1.16 | A **decision tree**: which pattern should I reach for? |
| R1.17 | **Pitfalls, costs, and best practices** |
| R1.18 | **References** — documentation and beginner tutorials to go deeper |

A note before we start. This notebook contains **runnable Claude calls** (from R1.6 onward). They use a tiny toy example and the smallest, cheapest model, so running the whole notebook costs a fraction of a cent. Everything before R1.6 — data structures, JSON, classes, Pydantic — runs with **no API key and no internet at all**, because it is plain Python.

> **One promise for the whole course (Principle, stated once and obeyed everywhere): this course uses Anthropic Claude, and only Anthropic Claude.** There is no OpenAI here, no Gemini, no Llama. One provider, one SDK, one set of habits to master. We will explain *why* this is a deliberate teaching choice when we get to model selection in R1.9.

***
## R1.1 Why would a data analyst use an LLM at all?

Let us start with the honest question, because it deserves an honest answer. You are a data analyst. You have pandas, NumPy, matplotlib, statsmodels, scikit-learn — a toolkit that can load a million rows, fit a regression, and draw a chart. **Why would you ever hand part of your job to a language model?**

The answer is a single word: **language**. Almost everything a data analyst produces eventually has to turn into *sentences a human will read* — a Slack message, a memo to leadership, a caption under a chart, a paragraph in a report. That last mile, from a computed number to a clear sentence a busy executive understands in five seconds, is real work. It is slow, it is repetitive, and it is exactly the kind of work a Large Language Model is genuinely excellent at.

Think of the analyst's day as two very different kinds of task:

**Things computers have always been good at — arithmetic and logic.**
- "What is the mean click-through rate across these three ad variants?"
- "Is the difference between variant A and variant B statistically significant?"
- "Sort these eleven predictors by the absolute value of their correlation with quality."

These are *calculations*. Python answers them perfectly, exactly, and reproducibly. You would never ask a language model to do arithmetic — it might get it wrong, and you would have no way to check.

**Things computers were historically bad at — turning structure into fluent prose.**
- "Write two sentences a non-technical marketing lead can act on, explaining which ad won and by how much."
- "Summarise these findings as three bullet points, each naming the caveat honestly."
- "Rephrase this raw statistical result so a winemaker — not a statistician — understands it."

These are *language* tasks. This is the new ability. A Large Language Model (an **LLM** — a program trained on an enormous amount of text to predict and produce fluent, context-appropriate language) is, for the first time, a tool you can call from your code to do the *writing* part of analysis.

### LLMs augment the analyst — they never replace the analysis

Here is the framing we will hold onto for the entire course, and it is worth saying slowly because it is easy to get wrong in the excitement:

> **The LLM does not do your analysis. You do your analysis. The LLM helps you *communicate* it.**

A useful mental image: the LLM is a brilliant, fast, tireless **junior copywriter** who sits next to you. This copywriter is fantastic at turning your findings into clean prose — but they cannot be trusted to do the maths, and they have never seen your data. So you do the maths yourself, in Python, and you hand the copywriter the finished numbers along with clear instructions. They write the memo; you check it; you ship it.

That division of labour — *you and Python do the thinking and the counting; the model does the phrasing* — is not just a nice idea. It is the rule that keeps your work trustworthy. It is so important that it gets its own section, next.

***
## R1.2 The one rule: "Python computes, the LLM interprets"

If you remember one sentence from this entire notebook, remember this one:

> ### Python computes the numbers. The LLM only interprets the numbers Python already computed.

We never let the model invent a figure. Not a mean, not a percentage, not a p-value, not a count. Every number that ends up in a sentence the model writes was **calculated in Python first** and then **handed to the model as part of the prompt**. The model's only job is to wrap those exact numbers in good English.

### Why this rule exists

LLMs generate text by predicting what words tend to follow other words. That is astonishingly powerful for *language*, but it means a model can produce a number that *sounds* right and is simply wrong — a phenomenon people call a **hallucination** (the model confidently states something that is not true). If you ask a model "what's the average of 5.0%, 6.5%, and 5.8%?", it will usually be right — but "usually" is not good enough when a stakeholder is about to make a decision based on the figure, and you have no automatic way to catch the rare miss.

So we remove the possibility entirely. We never ask the model to calculate. We calculate in Python — which is exact, reproducible, and checkable — and we pass the finished numbers in. The model is then *physically unable* to get the arithmetic wrong, because it is not doing any arithmetic. It is only choosing words.

### What this looks like in practice

Every Claude call in this course follows the same three-beat rhythm:

1. **Python computes.** You run real code on real data and get real numbers into Python variables.
2. **You build the prompt.** You write a message to Claude that *contains those exact numbers* and says, in plain language, what you want written.
3. **The model interprets.** Claude returns sentences. The sentences contain your numbers, dressed in prose for your audience.

We will see this rhythm three times in this notebook — once for each of the three "patterns" of calling Claude. But the rhythm never changes, and the rule above never bends. Whenever you are tempted to ask the model "and what's the percentage?", stop: that is Python's job. Hand the model the percentage.

We are about to need to *carry numbers* from Python into a prompt, and to *read structured answers* back out. To do that comfortably, we need three small Python building blocks first. Let us meet them.

***
## R1.3 Three Python building blocks: lists, dictionaries, tuples

Before we can talk to an API, we need to be fluent in the three containers Python uses to hold collections of values. You have seen all three in passing in Sessions 01–04; here we define each one **from scratch**, with a tiny concrete example, because the entire machinery of API calls is built out of them.

A quick framing first. A plain variable holds **one** value: `ctr = 0.065` holds a single number. But analysis is full of *collections* — three ad variants, eleven wine measurements, a row of fields. For collections we need containers. Python's three everyday containers are the **list**, the **dictionary**, and the **tuple**. They differ in two questions: *Is the order meaningful?* and *Do I look things up by position, or by name?*

### The list — an ordered shelf of items

A **list** is an ordered sequence of values, written with square brackets `[...]`. "Ordered" means each item has a position, counted **from zero** (Python, like most programming languages, starts counting at 0, not 1). You reach an item by its position number, called its **index**.

### The dictionary — a labelled set of pigeonholes

A **dictionary** (or **dict**) stores values not by position but by **name**. It is written with curly braces `{...}`, and each entry is a `key: value` pair. The *key* is the label you look things up by; the *value* is what is stored under that label. A dictionary is the single most important container for API work, because — as we will see in R1.4 — it maps almost perfectly onto the JSON format that APIs speak.

### The tuple — a fixed little group

A **tuple** is like a list — ordered, indexed from zero — but it is **immutable**, meaning once you create it you cannot change its contents. It is written with parentheses `(...)`. Tuples are perfect for a small group of values that naturally belong together and should not be edited, such as an `(impressions, clicks)` pair.

Let us make all three concrete with a tiny advertising example — the kind of data you met when we ran A/B tests in **Session 03**.

In [ ]:
# A LIST: three ad variants, in order. Square brackets, comma-separated.
variants = ["A", "B", "C"]

# Reach an item by its index (position). Python counts from 0, so index 0 is the first item.
print("First variant (index 0):", variants[0])

# Index 1 is the SECOND item, because counting starts at zero.
print("Second variant (index 1):", variants[1])

# len() tells us how many items the list holds.
print("Number of variants:", len(variants))

# A DICTIONARY: clicks recorded for each variant, looked up BY NAME, not by position.
# Curly braces; each entry is  key: value.  Here the key is the variant name, the value is a click count.
clicks = {"A": 50, "B": 65, "C": 58}

# Look a value up by its KEY (the label), using square brackets with the key inside.
print("Clicks for variant B:", clicks["B"])

# A TUPLE: a fixed (impressions, clicks) pair for variant A. Parentheses; cannot be changed later.
variant_a = (1000, 50)

# Tuples are indexed just like lists: index 0 is impressions, index 1 is clicks.
print("Variant A impressions:", variant_a[0], "| clicks:", variant_a[1])

Read the output above against the code. Three things to lock in, because every API call leans on them:

1. **Lists are ordered and indexed from zero.** `variants[0]` is `"A"`, the *first* item. This off-by-one feeling fades quickly with practice.
2. **Dictionaries are looked up by key.** `clicks["B"]` gives `65` because `"B"` is the label we filed `65` under. We did not need to know "B is the second one" — we just asked for it by name.
3. **Tuples are fixed little groups.** `(1000, 50)` bundles two related numbers that we never intend to edit separately.

You will see all three constantly from here on. In particular, **the message you send to Claude is a list of dictionaries, and Claude's reply is an object built out of lists and dictionaries.** Once you are comfortable with `[...]` and `{key: value}`, the API stops looking mysterious. Next we meet the text format that dictionaries turn into when they travel across the internet: JSON.

***
## R1.4 JSON — how structured data travels as text

Here is a problem. Inside your running Python program, a dictionary like `{"A": 50, "B": 65}` lives in your computer's memory as a Python object. But an API call has to send data **across the internet** to Anthropic's servers and receive data back. The internet does not move "Python objects" around — it moves **plain text**. So we need an agreed way to write a dictionary (or a list, or a number) as a *string of text* that any program, in any language, can read back and reconstruct.

That agreed way is **JSON** — **J**ava**S**cript **O**bject **N**otation. Despite the "JavaScript" in the name, JSON is a universal, language-neutral text format. It is, by a wide margin, the most common way for APIs to exchange structured data, and Anthropic's API uses it for everything.

### The good news: JSON looks almost exactly like Python

If you can read a Python dictionary and a Python list, you can already read JSON, because the notation is nearly identical:

- A JSON **object** looks like a Python dict: `{"variant": "B", "clicks": 65}`
- A JSON **array** looks like a Python list: `["A", "B", "C"]`
- JSON **strings** use double quotes: `"hello"` (single quotes are not allowed in JSON — a common beginner trip-up)
- JSON **numbers** look like Python numbers: `65`, `0.065`
- JSON **booleans** are lowercase: `true` and `false` (Python writes `True`/`False` — Python's `json` library translates this for you)
- JSON's "nothing here" value is `null` (Python's `None`)

The handful of small differences (double quotes only, lowercase `true`/`false`, `null` instead of `None`) are exactly why we do **not** convert by hand. Python ships with a built-in library called **`json`** that does the translation in both directions, perfectly, every time.

### Two verbs to remember: `dumps` and `loads`

The `json` library gives us two functions we will use forever:

- **`json.dumps(obj)`** — "**dump** to **s**tring": take a Python object (dict, list, …) and produce its JSON **text** representation. Use this when you are *sending* data out.
- **`json.loads(text)`** — "**load** from **s**tring": take a JSON **text** string and reconstruct the Python object. Use this when you are *reading* data in.

A memory hook: the `s` on the end of each name stands for **string**. `dumps` = "dump to string", `loads` = "load from string". Let us watch a Python dictionary make the round trip out to text and back.

In [ ]:
# Import Python's built-in json library. "Built-in" means it ships with Python — no installation needed.
import json

# Start with an ordinary Python dictionary describing one ad variant.
result = {"variant": "B", "clicks": 65, "impressions": 1000, "is_winner": True}

# json.dumps(...) turns the Python object into a JSON TEXT string.
# indent=2 just pretty-prints it with 2-space indentation so it is easy to read.
json_text = json.dumps(result, indent=2)

# Print the JSON text. Notice: double quotes everywhere, and Python's True became JSON's lowercase true.
print("As JSON text (this is what travels across the internet):")
print(json_text)

# Confirm it really is text now: its Python type is str (a string).
print("\nType of json_text:", type(json_text))

# Now the round trip back: json.loads(...) reconstructs a Python object FROM the JSON text.
back_to_python = json.loads(json_text)

# We have a real dictionary again — so we can look values up by key, exactly as in R1.3.
print("\nReconstructed Python dict. Clicks for the winner:", back_to_python["clicks"])

# Confirm the type is back to dict.
print("Type after json.loads:", type(back_to_python))

Look carefully at the output and you have understood JSON completely:

- The dictionary went **out** to text with `json.dumps`. In the printed text, every key is in double quotes and Python's `True` was automatically written as JSON's lowercase `true`. That text string is exactly the sort of payload that crosses the internet to Anthropic and back.
- The text came **back** to a Python dictionary with `json.loads`, and we immediately looked up `back_to_python["clicks"]` by key — proof it is a normal dict again.

Why does this matter for talking to Claude? Two reasons, one for each direction:

1. **Sending:** when you call the API, the SDK packages your message as JSON and ships it off. You will mostly not see this happen — the library does it — but it is good to know the message you build *becomes* JSON under the hood.
2. **Receiving:** in **Pattern 2** (R1.12) we will deliberately ask Claude to reply *in JSON*, and then use `json.loads` to turn its text answer into a Python dictionary we can pull fields out of. That is the moment this section pays off.

We now have the data-handling vocabulary — lists, dicts, tuples, and JSON. It is finally time to answer the question the whole notebook is named after: what *is* an API, and what happens when we "call" one?

***
## R1.5 What is an API? (no jargon, promise)

The letters stand for **A**pplication **P**rogramming **I**nterface, which is a mouthful that explains nothing. So forget the letters and picture a **restaurant**.

You sit at a table. You do not walk into the kitchen and cook. Instead, there is a **menu** listing exactly what you can order and how to order it, and there is a **waiter** who carries your order to the kitchen and brings the food back. You never see the kitchen. You do not need to. The menu-and-waiter system is a clean, agreed boundary between *you* (who wants a meal) and *the kitchen* (which knows how to make one).

**An API is the menu-and-waiter for a piece of software.** It is a published, agreed list of things you are allowed to ask a remote service to do, and the agreed way to ask. You send a request that follows the menu; the service does the work on its own powerful machines; it sends a response back. You never see the inside of Anthropic's servers — you do not need to. You just place valid orders.

### The request → response cycle

Every API interaction is the same two-step dance, called the **request–response cycle**:

1. **You send a request.** "Here is my order: this model, this message, this maximum length." Your request travels over the internet using the same protocol your web browser uses to load pages — **HTTP** (Hyper-Text Transfer Protocol; the basic language computers use to ask each other for things over the web).
2. **The service sends a response.** Anthropic's servers run the model on your request and send back a reply — as JSON text, naturally. Your code reads it.

That is genuinely the whole concept. A request goes out, a response comes back, both as structured text.

### Where do the numbers in the request and response live? In JSON, of course

Now you see why we did R1.3 and R1.4 first. The **request** you send is, underneath, a JSON object containing a *list* of message *dictionaries*. The **response** that comes back is JSON too, which the SDK hands you as a tidy Python object made of — you guessed it — lists and dictionaries. Everything you learned in the last two sections is the raw material of every API call.

### The one secret ingredient: the API key

There is one more thing the restaurant analogy needs. To order from this particular kitchen, you must prove you are an account holder, because Anthropic charges for the compute you use. You do this with an **API key** — a long secret string of characters, unique to your account, that you include with every request. Think of it as a membership card the waiter checks. Two rules about it, both important:

- **Keep it secret.** Anyone with your key can spend money on your account. Never paste it into a notebook, never commit it to git, never share it in a screenshot.
- **Never hard-code it.** Instead we store it in an **environment variable** (a setting that lives in your operating system, outside your code) called `ANTHROPIC_API_KEY`, and the SDK reads it from there automatically. Your code never contains the secret — it just trusts that the environment has it.

We arranged exactly this in the course setup (the repository README walks through it). In the next section we will confirm your key is in place, and if it is missing we will **stop the notebook immediately with a clear message** rather than limp forward and fail confusingly later. Let us set up the SDK.

***
## R1.6 Setting up the Anthropic Python SDK

To talk to Claude from Python we use Anthropic's official **SDK** (**S**oftware **D**evelopment **K**it — a ready-made library that wraps all the HTTP-and-JSON plumbing from R1.5 so you can write one clean Python line instead of hand-assembling web requests). The library is called **`anthropic`**, and it is already installed in this course's `ailab` environment. If you ever needed to install it yourself, the command would be `pip install anthropic` — but you do not need to run it here.

We will do the setup in two careful steps, each in its own cell:

1. **Confirm the API key is present** — and stop loudly if it is not.
2. **Create the client** — the single object through which every call to Claude flows.

### Step 1 — Confirm the API key, and hard-stop if it is missing

Following the rule from R1.5, your secret key lives in an environment variable named `ANTHROPIC_API_KEY`, never in the code. The cell below reads that variable. If it is empty, we deliberately **halt the whole notebook** with an actionable message. This is much kinder than letting you run twenty more cells only to hit a cryptic authentication error at the very end.

In [ ]:
# Import os, Python's standard library for talking to the operating system (including environment variables).
import os

# Read the environment variable ANTHROPIC_API_KEY. os.environ.get(...) returns None if it is not set.
anth_key = os.environ.get("ANTHROPIC_API_KEY")

# If the key is missing, stop the notebook right here with a clear, actionable message.
if not anth_key:
    raise SystemExit(
        "ANTHROPIC_API_KEY is not set in your environment.\n"
        "Follow the API-key step in the repository README, fully close VS Code, reopen it, "
        "and re-run this cell. Every section from here on needs the key."
    )

# The key is present. Print a confirmation WITHOUT revealing the secret itself — only its length.
print(f"ANTHROPIC_API_KEY is set. Key length: {len(anth_key)} characters.")

If that cell printed a confirmation, you are ready. If it stopped with the message instead, fix the key as described and re-run — nothing below will work until it passes.

### Step 2 — Create the client

The **client** is one Python object that represents your authenticated connection to Anthropic. You create it once, and then every request you make goes through it. When you write `anthropic.Anthropic()` with empty parentheses, the SDK quietly looks for `ANTHROPIC_API_KEY` in the environment and uses it — which is exactly why we never type the key into the code.

In [ ]:
# Import the official Anthropic SDK we just discussed.
import anthropic

# Create the client object. With empty parentheses it automatically reads ANTHROPIC_API_KEY from the environment.
client = anthropic.Anthropic()

# Print a confirmation that the client object now exists and is ready to make calls.
print("Anthropic client ready.")

One object, `client`, now holds your connection to Claude. Every single call in the rest of this notebook is a method on this object — specifically `client.messages.create(...)`. That method is the menu item we will order again and again, so we are going to dissect it one argument at a time in the next section before we ever run it for real.

***
## R1.7 The anatomy of `client.messages.create(...)`

Here is the single most important line in the whole course. Every time we talk to Claude, it is some version of this:

```python
response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=300,
    system="You are a helpful data-analysis assistant.",
    messages=[
        {"role": "user", "content": "Hello, Claude!"}
    ],
)
```

Let us name every part of it, slowly, because once this is clear, *nothing* in the API will surprise you again.

- **`client.messages.create(...)`** — the method we call. Read it left to right: on our `client`, reach into its `messages` capability, and `create` a new message (Claude's reply). This is us placing the order with the waiter from R1.5.

- **`model=`** — *which* Claude should answer. Models differ in capability, speed, and cost; we use the name string `"claude-haiku-4-5"`, the small, fast, inexpensive model that has been this course's default since Session 01. (R1.9 is all about how to choose.)

- **`max_tokens=`** — the *maximum length* of Claude's reply, measured in **tokens**. A **token** is a chunk of text a bit smaller than a word — very roughly, one token is about ¾ of an English word, or about four characters. `max_tokens=300` says "do not write more than ~300 tokens back." It is a safety cap (it stops a runaway answer and bounds your cost); Claude will usually stop naturally well before it.

- **`system=`** — the **system prompt**: a standing instruction that sets Claude's *role and rules* for the whole conversation, separate from the specific question. "You are a Drivers Analyst Assistant who never invents numbers" is a system prompt. Think of it as the job description you hand the junior copywriter before they start.

- **`messages=`** — the actual conversation, and here is where R1.3 pays off: it is a **list** of **dictionaries**. Each dictionary is one turn and has exactly two keys:
    - **`"role"`** — who is speaking. For now, `"user"` (that is you). Claude's own turns have the role `"assistant"`.
    - **`"content"`** — what is said, as text.

So `messages=[{"role": "user", "content": "Hello, Claude!"}]` reads, in plain English, as: *"Here is a one-turn conversation; the user said 'Hello, Claude!'"* Claude's job is to produce the next turn.

### The two layers of instruction: `system` vs `messages`

Beginners often ask why instructions are split between `system` and the user message. The clean rule:

- **`system`** = *who Claude is and the rules it must always follow* — the persona and the guardrails. It does not change from question to question.
- **the user `content`** = *this specific request, including the actual numbers Python computed.* It changes every call.

Keeping the standing rules in `system` and the specific data in the user message is tidy, and it makes the "Python computes, the LLM interprets" rule (R1.2) easy to honour: the computed numbers go in the user message, every time.

Enough theory. Let us send the very first real request and watch a reply come back.

In [ ]:
# Call Claude for the first time. We assign whatever comes back to the variable `response`.
response = client.messages.create(
    # Which model answers — the small, fast, cheap course default.
    model="claude-haiku-4-5",
    # Cap the reply length at ~120 tokens. This is a short greeting, so we do not need many.
    max_tokens=120,
    # The system prompt: Claude's standing role for this call.
    system="You are a friendly assistant helping a junior data analyst learn the Anthropic API.",
    # The conversation so far: a list with ONE dictionary — one user turn.
    messages=[
        # role says who is speaking ("user" = us); content is what we say.
        {"role": "user", "content": "In one short sentence, what is your job in a data-analysis workflow?"}
    ],
)

# `response` is a rich object (we dissect it next). For now, reach in and print just the text Claude wrote.
# response.content is a LIST of blocks; the first block [0] is text; .text is the string inside it.
print(response.content[0].text)

You just made an API call. A request went out over HTTP carrying your model choice, your length cap, your system prompt, and your one-turn message; Anthropic ran the model; a JSON response came back; and the SDK handed it to you as the `response` object. We printed one piece of it — the text — with `response.content[0].text`.

That little chain of `.content[0].text` is doing a lot, and it deserves a proper explanation, because the response object is *not* just a string. It is a structured object with several useful parts. Let us open it up.

***
## R1.8 Reading the response object Claude sends back

When the call returns, you do **not** get a plain string. You get a structured **Message object**, because Anthropic sends back more than just the words — it also tells you which model answered, why it stopped, and how many tokens it used. Understanding the shape of this object means you never have to guess where the text is.

The parts you will use most:

- **`response.content`** — the heart of it. This is a **list of content blocks**. For an ordinary text reply the list has one block, of type `"text"`, and the words live in that block's `.text`. (In Pattern 3, R1.15, the list will instead contain a block of type `"tool_use"` — same structure, different block type. That is why `content` is a *list*: it can hold different kinds of blocks.)
- **`response.stop_reason`** — *why* Claude stopped. Usually `"end_turn"` (it finished naturally). If it says `"max_tokens"`, your reply was cut off because it hit the length cap — a sign to raise `max_tokens`.
- **`response.usage`** — a small object with `input_tokens` and `output_tokens`, the token counts for this call. Since you are billed per token, this is how you watch your costs (more in R1.17).
- **`response.model`** — the exact model version that answered.

Let us inspect the `response` we already have from R1.7, field by field, so the structure becomes concrete.

In [ ]:
# response.content is a LIST of content blocks. Let us see how many blocks came back (for plain text: just 1).
print("Number of content blocks:", len(response.content))

# Look at the FIRST block (index 0, because lists count from zero — R1.3).
first_block = response.content[0]

# Every block carries a .type telling us what kind it is. For an ordinary reply this is the string "text".
print("Type of the first block:", first_block.type)

# When the type is "text", the words live in the block's .text attribute. This is the actual reply.
print("\nThe text Claude wrote:")
print(first_block.text)

# stop_reason tells us WHY Claude stopped. "end_turn" = it finished naturally; "max_tokens" = it was cut off.
print("\nStop reason:", response.stop_reason)

# usage reports the token counts for this single call — what you are billed on.
print("Input tokens:", response.usage.input_tokens, "| Output tokens:", response.usage.output_tokens)

# model confirms exactly which model version produced this answer.
print("Answered by model:", response.model)

Now `response.content[0].text` from the previous section reads like plain English: *"take the response, look at its content list, take the first block, give me the text inside it."* No magic.

A habit worth forming early: after a call, glance at `stop_reason`. If it is `"end_turn"`, Claude said everything it wanted to. If it is `"max_tokens"`, the answer was truncated mid-thought and you should raise `max_tokens` and try again. And glance at `usage` now and then to keep a feel for what your calls cost.

We have now made a call and read its reply. Before we put this to work on real analyst tasks, two short conceptual sections will make every future call better: *which model to pick* (R1.9) and *how to write the prompt* (R1.10).

***
## R1.9 Choosing a model — Haiku, Sonnet, and Opus

When you wrote `model="claude-haiku-4-5"`, you made a choice. Anthropic offers a **family** of Claude models, and they trade off three things against each other: **capability** (how good the reasoning and writing are on hard tasks), **speed** (how fast the reply comes back), and **cost** (how much you pay per token). The family is named after three poetic forms, smallest to largest:

| Model | Personality | Best for |
|---|---|---|
| **Haiku** | Fast and inexpensive, like its three-line namesake. Very capable for everyday language tasks. | High-volume, latency-sensitive, well-scoped jobs: summarising, classifying, extracting fields, rephrasing a computed result. **This course's default.** |
| **Sonnet** | The balanced middle. Noticeably stronger reasoning than Haiku, still fast and affordable. | The majority of "serious" production work — multi-step reasoning, nuanced writing, trickier analysis prose. |
| **Opus** | The most capable and the most expensive. | The genuinely hard problems: long, intricate reasoning; complex multi-part instructions; the cases where quality matters more than cost. |

Within each tier the models carry **version numbers** — `claude-haiku-4-5` is "the 4.5 generation of Haiku." Anthropic releases new generations over time, and prices and exact model names change, so **we deliberately do not hard-code any prices in this notebook.** The authoritative, always-current lists are the model and pricing pages, both linked in R1.18 — check them when you start a real project.

### A simple rule for picking

> **Start with the smallest model that does the job. Move up only when the output is not good enough.**

For everything in this course — turning a handful of computed numbers into a clean memo paragraph or a small structured plan — **Haiku is more than enough**, which is why it is our default. You are not asking the model to reason through a hard problem; you are asking it to *write well about numbers you already computed* (R1.2). That is squarely in Haiku's wheelhouse, and it keeps your calls fast and nearly free. If one day you hand the model a genuinely complex, multi-step reasoning task and Haiku's answer disappoints, *that* is the moment to try Sonnet, then Opus.

### Why only Claude? A note on the course's single-vendor choice

You may be wondering why this course never reaches for a different vendor's model. This is a deliberate teaching decision, and there are three honest reasons:

1. **One SDK, one mental model.** For someone learning their first API, juggling two providers' libraries, two key names, and two sets of quirks doubles the confusion for no learning benefit. Mastering *one* provider deeply beats being half-fluent in two.
2. **Tool use already gives us what we need.** In R1.15 you will see that Anthropic's "tool use" feature enforces a strict output structure *at the moment of generation* — which is exactly the capability one might otherwise switch vendors to get. We do not need to leave to get schema-guaranteed output.
3. **A focused portfolio.** You will finish this course genuinely confident in one provider's patterns, which is far more useful in a job than a shallow tour of several.

If you later want to explore other providers for a personal project, wonderful — but in *this* course, it is Claude, end to end. Now, how do we ask Claude well?

***
## R1.10 Writing good prompts

A **prompt** is simply the text you send the model — your system prompt plus your user message. The quality of Claude's answer depends enormously on the quality of your prompt, and the good news is that "prompting well" is mostly a small set of common-sense habits, not a dark art. Here are the ones that matter most for an analyst:

1. **Be explicit about the role (in `system`).** "You are a marketing analytics assistant who writes for non-technical stakeholders" steers tone and vocabulary far better than leaving the role unsaid.
2. **State the task plainly and concretely (in the user message).** "Write two sentences" beats "write a bit about this." Say how long, for whom, and in what tone.
3. **Give the model the numbers — and forbid invention.** Per R1.2, paste the computed figures right into the prompt, and add an explicit instruction such as *"Use only the numbers given above; do not invent any figures."* This single sentence is your strongest guard against hallucinated statistics.
4. **Show the shape you want.** If you want three bullet points, ask for three bullet points. If you want JSON, say so and show the keys (that is Pattern 2, R1.12).
5. **Constrain the scope.** "Do not give recommendations beyond what the data supports" keeps the model honest and on-task.

You have, in fact, been seeing all five habits at work in the system prompts and user messages of every session's API call. Now you know *why* they were written that way.

### Go deeper: Anthropic's official Prompt Engineering material

Anthropic publishes excellent, free, beginner-friendly material devoted entirely to writing better prompts, and it is worth your time once you are comfortable with the mechanics in this notebook. There are two complementary pieces:

> **📘 Anthropic's Prompt Engineering resources**
>
> - **The Prompt Engineering Overview** (documentation) — a structured guide to the core techniques: *being clear and direct*, *giving examples*, *letting the model think step by step*, *using XML tags to mark up parts of a prompt*, *assigning a role with the system prompt*, and *chaining prompts together*. It is concise and practical. → linked in R1.18.
> - **The Interactive Prompt Engineering Tutorial** (a hands-on GitHub course) — nine chapters of exercises you actually run, each building one skill at a time, with a playground to experiment in. It uses Haiku, the same model we use here, so everything transfers directly. → linked in R1.18.
>
> A good moment to work through these is *right after* finishing this notebook: you will understand every line of code in the tutorial, so you can focus purely on the prompting craft.

Everything is now in place — data structures, JSON, the API concept, the SDK, the call anatomy, model choice, and prompting habits. Time to put it to work. We will learn the **three patterns** for calling Claude, in order of increasing structure and control. Pattern 1 is the simplest: ask a question, get a paragraph.

***
## R1.11 Pattern 1 — plain text in, plain text out

This is the simplest and most common pattern, and it is exactly what we did in **Session 01**: you send a message, Claude sends back a paragraph of prose, and you use that prose. Use this pattern whenever the *output is meant to be read by a human* — a memo, a Slack note, a chart caption — and you do not need to pull individual fields out of the answer programmatically.

Let us follow the three-beat rhythm from R1.2 on a tiny, verifiable example. The scenario is a callback to **Session 03**, where we ran A/B tests: marketing tried three ad variants and we want a two-sentence note for a non-technical lead.

### Beat 1 — Python computes (no model involved yet)

First, real arithmetic in Python. The **click-through rate (CTR)** of a variant is the fraction of people who clicked after seeing it:

$$\text{CTR} = \frac{\text{clicks}}{\text{impressions}}$$

where **clicks** is how many people clicked and **impressions** is how many people saw the ad. And the **relative lift** of a variant over the control — the same quantity we computed in Session 03 — is how much bigger its CTR is, *as a percentage of the control's CTR*:

$$\text{relative lift} = \left( \frac{\text{CTR}_{\text{variant}}}{\text{CTR}_{\text{control}}} - 1 \right) \times 100\%$$

The fraction $\text{CTR}_{\text{variant}} / \text{CTR}_{\text{control}}$ asks "how many times the control's rate is this variant?"; subtracting 1 turns "1.30 times as big" into "0.30 bigger"; multiplying by 100 expresses it as a percentage. Let Python do it.

In [ ]:
# Our tiny toy experiment: three ad variants, each shown to 1000 people. A is the control.
# We store impressions and clicks as dictionaries keyed by variant name (R1.3).
impressions = {"A": 1000, "B": 1000, "C": 1000}
clicks      = {"A": 50,   "B": 65,   "C": 58}

# Compute the click-through rate for each variant: clicks / impressions.
# A dictionary comprehension builds a new dict by looping over the variant names.
ctr = {v: clicks[v] / impressions[v] for v in impressions}

# Find the winning variant: the key whose CTR value is the largest. max(..., key=ctr.get) does exactly that.
winner = max(ctr, key=ctr.get)

# Compute the winner's relative lift over the control "A", as a percentage (the Session 03 formula above).
rel_lift = (ctr[winner] / ctr["A"] - 1) * 100

# Print every computed number so we can VERIFY them by eye before handing them to Claude.
for v in ctr:
    print(f"Variant {v}: CTR = {ctr[v]*100:.1f}%")
print(f"\nWinner: variant {winner} (CTR {ctr[winner]*100:.1f}%), "
      f"relative lift over control A = {rel_lift:+.0f}%")

Check the arithmetic by hand, because this is the whole point of R1.2: A is $50/1000 = 5.0\%$, B is $65/1000 = 6.5\%$, C is $58/1000 = 5.8\%$. The winner is **B**, and B's lift over A is $(6.5/5.0 - 1)\times 100 = +30\%$. **Python computed every one of these. The model will see them and not one figure more.**

### Beat 2 + 3 — build the prompt, let the model interpret

Now we hand those exact numbers to Claude and ask only for the *writing*. Notice three things in the call below: the numbers are pasted into the user message via an f-string; the system prompt sets the role and the no-invention rule; and we ask for a specific shape ("two sentences, plain language").

In [ ]:
# Build the message text by pasting the PYTHON-COMPUTED numbers into a normal Python string (an f-string).
# The model will only ever see numbers we put here — it computes nothing itself (R1.2).
user_message = (
    f"A/B/C ad test, 1000 impressions each.\n"
    f"Variant A (control): CTR {ctr['A']*100:.1f}%.\n"
    f"Variant B: CTR {ctr['B']*100:.1f}%.\n"
    f"Variant C: CTR {ctr['C']*100:.1f}%.\n"
    f"Winner: variant {winner}, with a relative lift of {rel_lift:+.0f}% over the control.\n\n"
    f"Write a two-sentence update for a non-technical marketing lead. "
    f"Use only the numbers above; do not invent any figures."
)

# Make the call. Plain text in, plain text out — Pattern 1.
note_response = client.messages.create(
    # The course-default small, fast, cheap model.
    model="claude-haiku-4-5",
    # A short note needs little room.
    max_tokens=200,
    # System prompt: role + the standing honesty rule.
    system=(
        "You are a marketing analytics assistant. You write short, clear updates for non-technical "
        "stakeholders. You never invent numbers; you use only the figures provided to you."
    ),
    # One user turn carrying our computed numbers and the writing instruction.
    messages=[{"role": "user", "content": user_message}],
)

# Pattern 1 reply is plain text: pull it out with .content[0].text, exactly as in R1.7/R1.8.
print(note_response.content[0].text)

Read Claude's note and confirm: every figure in it (5.0%, 6.5%, 5.8%, +30%, variant B) is one *we* computed and handed over. The model contributed the *sentences*, not the *statistics*. That is Pattern 1 working exactly as designed — and it is the right tool whenever the deliverable is prose for a human to read.

But sometimes you do not want prose. Sometimes you want the model to return **structured data** — distinct fields you can store in a database, drop into a table, or feed into the next step of a pipeline. For that we need the model's answer to come back in a shape your code can take apart. That is Pattern 2.

***
## R1.12 Pattern 2 — ask for JSON, then validate it yourself

Suppose that instead of a paragraph you want three separate, labelled pieces of information back — say a one-line `headline`, a `winner`, and a `recommendation` — so your code can store each field on its own. The natural move is to ask Claude to reply **in JSON** (R1.4), and then use `json.loads` to turn that reply into a Python dictionary you can pull fields out of.

This is **Pattern 2**, and it is exactly what we did in **Session 02**. The recipe has two new ingredients on top of Pattern 1:

1. **Ask for JSON explicitly, and name the keys you want.** Spell out the exact field names in the prompt, so the model knows the shape to produce.
2. **Use "assistant prefill" to force the reply to start as JSON.** Here is a genuinely useful trick: the `messages` list can end with a partial **assistant** turn, and Claude will *continue from where it left off*. If we make the last turn an assistant turn whose content is just an opening brace `{`, Claude is nudged to continue with valid JSON rather than chatty preamble like "Sure! Here's your JSON:". We then glue that `{` back on when we read the reply.

After the reply comes back, **we validate it ourselves** with `json.loads`. If the model produced malformed JSON, `json.loads` raises an error and we know immediately — the checking is *our* job in this pattern. (In Pattern 3 the checking moves to generation time and becomes automatic; that is the upgrade.) Let us see it.

In [ ]:
# Reuse the SAME computed numbers from R1.11 — Python already did the arithmetic; we never recompute via the model.
user_message_json = (
    f"A/B/C ad test results. Control A CTR {ctr['A']*100:.1f}%, B CTR {ctr['B']*100:.1f}%, "
    f"C CTR {ctr['C']*100:.1f}%. Winner: variant {winner}, relative lift {rel_lift:+.0f}% over control.\n\n"
    f"Reply with a JSON object and NOTHING else. Use exactly these three keys:\n"
    f'  "headline"      : a punchy one-line summary (string)\n'
    f'  "winner"        : the winning variant letter (string)\n'
    f'  "recommendation": one sentence on what to do next (string)\n'
    f"Use only the numbers above; do not invent figures."
)

# Make the call. The new trick is the SECOND message: a partial ASSISTANT turn containing just "{".
json_response = client.messages.create(
    model="claude-haiku-4-5",                       # same cheap default model
    max_tokens=300,                                 # a small JSON object needs little room
    system="You are a precise assistant that returns strictly valid JSON when asked. You never invent numbers.",
    messages=[
        # Our request, as the user turn.
        {"role": "user", "content": user_message_json},
        # PREFILL: we start Claude's OWN reply with "{" so it continues as JSON, not chit-chat.
        {"role": "assistant", "content": "{"},
    ],
)

# Because we prefilled "{", Claude's reply text is the REST of the JSON. Glue the "{" back on the front.
raw_json_text = "{" + json_response.content[0].text

# Show the raw text the model produced, so we can see it really is JSON.
print("Raw JSON text returned by Claude:")
print(raw_json_text)

# NOW WE VALIDATE IT OURSELVES. json.loads turns the text into a Python dict — or raises if it is malformed.
parsed = json.loads(raw_json_text)

# Success: it parsed. Pull the fields out BY KEY (R1.3) — proof we now have structured, usable data.
print("\nParsed successfully. Individual fields:")
print("  headline       :", parsed["headline"])
print("  winner         :", parsed["winner"])
print("  recommendation :", parsed["recommendation"])

That worked, and it is genuinely useful: we now hold three separate fields (`parsed["headline"]`, `parsed["winner"]`, `parsed["recommendation"]`) instead of one undifferentiated paragraph. We could write each to its own column in a results table.

But notice the soft spot. **Nothing guaranteed the model would return valid JSON, or the right keys.** It usually does — but "usually" is the same word that worried us about hallucinated numbers. If the model wrapped its JSON in an apology, or renamed `winner` to `winning_variant`, or forgot a field, our `json.loads` (or the key lookups after it) would blow up at runtime. In Pattern 2, **catching that is our responsibility**, and on a bad day it fails *after* the call, costing us a retry.

Wouldn't it be better if the structure were **guaranteed at the moment of generation** — if the model were *unable* to return the wrong shape? That is precisely what **Pattern 3** delivers, using Anthropic's *tool use* feature together with a Python library called *Pydantic*. But Pydantic is built on Python's **class** system, and to understand Pydantic you must first understand classes and objects. That is our next stop — and it is one we will not skip.

***
## R1.13 Python classes and objects — the object model, from scratch

We are about to meet **Pydantic**, the tool that makes Pattern 3 robust. But Pydantic is built directly on a feature of Python we have used *implicitly* all course without ever naming it: **classes**. So we stop here and build the idea from absolute zero. If you have never written `class` in your life, this section is for you. If you have, it is a five-minute refresher that will make Pydantic feel obvious.

### The motivation: some data comes in natural bundles

Think back to **Session 04**, where each wine had an `alcohol` level, a `density`, and a `quality` score. Those three numbers are not independent facts floating around — they *belong together*; they describe **one wine**. We could keep them in three separate lists and promise ourselves to always use the same index in each, but that is fragile and error-prone. It would be far nicer to have a single "wine" thing that carries its own alcohol, density, and quality together, as a unit.

That "thing that bundles related data (and the operations on it) into one unit" is the central idea of **object-oriented programming (OOP)**. Two words carry the whole concept:

- A **class** is a **blueprint** — a definition of what every thing of this kind has and can do. "A Wine has an alcohol level, a density, and a quality score." The blueprint itself is not a wine; it is the *design* for wines.
- An **object** (also called an **instance**) is **one actual thing built from the blueprint** — one specific wine, with its own particular numbers. From one `Wine` blueprint you can build thousands of wine objects, each with different values.

The relationship is exactly *cookie-cutter to cookies*, or *architect's drawing to houses*. One blueprint; many objects.

### The vocabulary, named once

- **Attributes** — the *data* an object carries (a wine's `alcohol`, `density`, `quality`). You read an attribute with a dot: `my_wine.alcohol`.
- **Methods** — the *functions that belong to an object*, defining what it can *do*. You call a method with a dot and parentheses: `my_wine.describe()`.
- **`__init__`** — a special method, pronounced "dunder init" ("dunder" = "double underscore"), that Python runs **automatically** every time you build a new object. Its job is to set up the new object's starting attributes. It is the "constructor."
- **`self`** — inside a class's methods, `self` is the placeholder for *"this particular object."* When you write `self.alcohol = alcohol`, you are saying "store the given alcohol value on *this specific wine I am building right now*." Every method's first parameter is `self`, and Python passes it in for you automatically.

You have *already been using* all of this. Every time you wrote `df.head()` or `df.mean()` in pandas, `df` was an **object** (an instance of pandas' `DataFrame` class) and `.head()` was a **method**. When you wrote `response.content` a few sections ago, `response` was an object and `content` was an **attribute**. OOP has been under your hands the whole time; now we name it. Let us build a class of our own.

In [ ]:
# Define a class — a BLUEPRINT for wines. By convention class names use CapitalisedWords.
class Wine:
    # __init__ runs automatically whenever we build a new Wine. It sets up the new object's attributes.
    # `self` is "this particular wine"; the other parameters are the values we pass in when building one.
    def __init__(self, name, alcohol, quality):
        # Store each incoming value as an ATTRIBUTE on this specific object (self).
        self.name = name            # this wine's name
        self.alcohol = alcohol      # this wine's alcohol % by volume
        self.quality = quality      # this wine's panel quality score

    # A METHOD: a function that belongs to the object and can use its own attributes via self.
    def describe(self):
        # Build and return a human-readable sentence from THIS wine's attributes.
        return f"{self.name}: {self.alcohol}% alcohol, panel quality {self.quality}."

# Confirm the blueprint is defined (no object built yet — this is just the design).
print("Class Wine defined (a blueprint, not yet a wine).")

In [ ]:
# Build an OBJECT (an instance) from the blueprint by calling the class like a function.
# The values we pass go straight into __init__ as name, alcohol, quality.
wine1 = Wine("Reserva Tinto", 13.5, 7)

# Build a SECOND, independent object. Same blueprint, different data.
wine2 = Wine("Table Red", 11.2, 5)

# Read an ATTRIBUTE with a dot. Each object carries its OWN values.
print("wine1's alcohol:", wine1.alcohol)
print("wine2's alcohol:", wine2.alcohol)

# Call a METHOD with a dot and parentheses. Note each wine describes ITSELF, using its own attributes.
print(wine1.describe())
print(wine2.describe())

# Proof they are independent: changing one object's attribute does not touch the other.
wine1.quality = 8                       # re-score wine1 only
print("After re-scoring wine1:")
print("  wine1 quality:", wine1.quality, "| wine2 quality:", wine2.quality)

Sit with that output for a moment, because it contains the whole idea:

- **One blueprint, two objects.** `Wine` is the class; `wine1` and `wine2` are two independent objects built from it, each carrying its own `name`, `alcohol`, and `quality`.
- **Attributes are read with a dot.** `wine1.alcohol` gave `13.5`; `wine2.alcohol` gave `11.2`. The same attribute *name*, different *values*, because they live on different objects.
- **Methods act on their own object.** `wine1.describe()` used `wine1`'s numbers; `wine2.describe()` used `wine2`'s. Inside `describe`, `self` was whichever wine we called it on.
- **Objects are independent.** Re-scoring `wine1` left `wine2` untouched.

This is the entire object model you need. To summarise the dot notation, which is the thing you will use constantly:

| You write | It means |
|---|---|
| `wine1.alcohol` | read the **attribute** `alcohol` (a piece of data) on object `wine1` |
| `wine1.describe()` | call the **method** `describe` (an action) on object `wine1` |

Now here is the punchline that ties this section to the next: **a Pydantic model is just a special kind of class** — one where you declare the attributes and their *types* up front, and Pydantic automatically checks that any data you put in actually matches. That automatic checking is exactly what will make Pattern 3 bulletproof. Let us meet Pydantic.

***
## R1.14 Pydantic — turning a class into a data contract

In R1.13 our `Wine` class happily accepted *anything*. We could have written `Wine("Oops", "not a number", 7)` and Python would not have complained until something broke much later. For a wine demo that is fine. For data arriving from an LLM, where we want to *guarantee* the shape, we want something stricter: a class that **enforces its own structure** and rejects bad data on the spot.

That is **Pydantic**. Pydantic is a Python library (already installed in `ailab`, because the `anthropic` SDK depends on it) for defining **data models** — classes that know the *type* of each field and validate data against those types automatically. A Pydantic model is a contract: *"an object of this kind has exactly these fields, of exactly these types."*

We use **Pydantic version 2** (the current major version). The three pieces you need:

- **`BaseModel`** — the Pydantic class you inherit from. "Inherit from" means *build your class on top of it* so it gets all of Pydantic's validation machinery for free. You write `class DriversPlan(BaseModel):` — read as "DriversPlan is a kind of BaseModel."
- **Type-annotated fields** — instead of writing an `__init__` by hand (as we did for `Wine`), you just *declare each field and its type*: `key_findings: str` means "this model has a field called `key_findings`, and it must be a string." Pydantic writes the `__init__` and the validation for you.
- **`Field(description=...)`** — an optional helper to attach a human-readable description (and other rules) to a field. Those descriptions become important in R1.15: they are the instructions Claude reads about what to put in each field.

Let us define a small model and watch it both *accept* good data and *reject* bad data.

In [ ]:
# Import the two Pydantic pieces we need. (Pydantic is installed as a dependency of the anthropic SDK.)
from pydantic import BaseModel, Field

# Define a Pydantic MODEL — a class that inherits from BaseModel, so it gains automatic validation.
# We just DECLARE each field with its type. No __init__ needed; Pydantic builds it for us.
class AdVerdict(BaseModel):
    # A text field: the one-line summary. The ": str" says this field MUST be a string.
    headline: str = Field(description="A punchy one-line summary of the A/B test result.")
    # Another text field: the winning variant letter.
    winner: str = Field(description="The winning variant letter, e.g. 'B'.")
    # A NUMBER field: the relative lift as a percent. ": float" says this MUST be a number.
    relative_lift_pct: float = Field(description="The winner's relative lift over control, in percent.")

# Build an object the normal way (keyword arguments matching the field names). Good data — this succeeds.
verdict = AdVerdict(headline="Variant B wins", winner="B", relative_lift_pct=30.0)

# Read fields with a dot, exactly like any object's attributes (R1.13).
print("headline          :", verdict.headline)
print("winner            :", verdict.winner)
print("relative_lift_pct :", verdict.relative_lift_pct)

In [ ]:
# Now watch Pydantic ENFORCE the contract. We try to build an object with a non-numeric lift.
# We wrap it in try/except so the validation error is caught and printed instead of stopping the notebook.
try:
    # "thirty" is text, but relative_lift_pct is declared as a float — this should be rejected.
    bad = AdVerdict(headline="Variant B wins", winner="B", relative_lift_pct="thirty")
except Exception as error:
    # Pydantic raises a ValidationError explaining exactly which field failed and why.
    print("Pydantic REJECTED the bad data, as it should:")
    print(error)

This is the superpower. The first cell built a valid `AdVerdict` and we read its fields with a dot. The second cell tried to slip in `"thirty"` where a number was required, and Pydantic **refused** with a precise error naming the offending field. Our `Wine` class from R1.13 would have silently accepted that garbage. Pydantic does not — it is a contract that checks itself.

Two more methods you will use, and then we have everything for Pattern 3:

- **`.model_dump()`** — turn a model *object* back into an ordinary Python **dictionary** (the reverse of building one). Handy for printing, saving, or passing along.
- **`.model_json_schema()`** — and this is the one that unlocks Pattern 3 — turn the model *class* into a **JSON Schema**: a JSON description of the model's structure (its field names, types, and descriptions). It does not describe one object; it describes *the shape every object must have*. This is precisely the format Anthropic's tool-use feature wants in order to constrain the model's output.

In [ ]:
# .model_dump() converts a model OBJECT into a plain Python dict (R1.3) — useful for saving or printing.
print("verdict.model_dump() gives a plain dict:")
print(verdict.model_dump())

# .model_json_schema() converts the model CLASS into a JSON SCHEMA describing the required shape.
# This schema — field names, types, and our Field descriptions — is what we hand to Claude in R1.15.
print("\nAdVerdict.model_json_schema() describes the SHAPE all AdVerdicts must have:")
print(json.dumps(AdVerdict.model_json_schema(), indent=2))

Read that schema. Notice it contains the field names (`headline`, `winner`, `relative_lift_pct`), each field's `type` (`string`, `string`, `number`), our human-readable `description`s, and a `required` list. It is a complete, machine-readable specification of "what a valid `AdVerdict` looks like" — and crucially, it is itself just JSON (R1.4).

Now hold these two facts side by side:

1. We can describe a required output shape as a JSON Schema (`.model_json_schema()`).
2. Anthropic's API has a feature — **tool use** — that accepts a JSON Schema and constrains Claude to produce output matching it *at generation time*.

Put them together and you get **Pattern 3**: schema-guaranteed structured output, with the validation moved from *after* the call (Pattern 2's weak spot) to *during* the call. Let us build it.

***
## R1.15 Pattern 3 — tool use + Pydantic (the robust pattern)

This is the pattern we used in **Session 03** (§3.12) and again in **Session 04** (§4.16), and now you have every piece needed to understand it completely. It solves Pattern 2's weakness: instead of *hoping* Claude returns the right JSON and checking afterward, we *tell the API the exact shape required* and Claude is constrained to produce it.

The mechanism is a feature called **tool use**. The name is slightly misleading for our purposes, so here is the honest framing: Anthropic designed "tool use" so Claude can call functions you define (a weather lookup, a database query, …). But it has a wonderful side effect we exploit: **to call a tool, Claude must produce arguments that match the tool's declared input schema** — and that schema can be *our Pydantic model's JSON Schema*. So we define a "tool" that does nothing but "submit a result in this exact shape," force Claude to call it, and read the perfectly-structured arguments back out. We are using tool use as a **structured-output guarantee**.

Three new ingredients on top of the earlier patterns:

1. **`tools=[...]`** — a list describing the tool(s) Claude may call. Each is a dictionary with three keys: `"name"`, `"description"`, and `"input_schema"`. For `input_schema` we plug in `OurModel.model_json_schema()` from R1.14 — the schema *is* the contract.
2. **`tool_choice={"type": "tool", "name": ...}`** — this *forces* Claude to call our specific tool rather than replying with free text. With this set, Claude has no option but to produce arguments matching our schema.
3. **Reading a `tool_use` block.** Recall from R1.8 that `response.content` is a *list of blocks*. With tool use, the block we want has `block.type == "tool_use"`, and its already-structured arguments live in `block.input` — a plain Python dict that conforms to our schema. We then wrap it back into our Pydantic model for type-safe access.

Let us reuse our A/B test numbers and produce a guaranteed-shape `AdVerdict`.

In [ ]:
# Build the "tool" Claude may call. Its input_schema IS our Pydantic model's JSON Schema (R1.14).
# Claude can only "call" this tool by producing arguments that match that schema — that is our guarantee.
verdict_tool = {
    "name": "submit_ad_verdict",                                            # the tool's name
    "description": "Submit the A/B test verdict. Fill in every field using only the numbers provided.",
    "input_schema": AdVerdict.model_json_schema(),                          # the shape, straight from Pydantic
}

# Build the user message with the SAME Python-computed numbers (R1.2 — the model still invents nothing).
verdict_message = (
    f"A/B/C ad test. Control A CTR {ctr['A']*100:.1f}%, B CTR {ctr['B']*100:.1f}%, "
    f"C CTR {ctr['C']*100:.1f}%. Winner: variant {winner}, relative lift {rel_lift:+.0f}% over control. "
    f"Submit the verdict using only these numbers."
)

# Make the call, now WITH tools and a forced tool_choice.
verdict_response = client.messages.create(
    model="claude-haiku-4-5",                                               # same cheap default model
    max_tokens=400,                                                         # room for the structured arguments
    system="You translate computed A/B-test numbers into a structured verdict. You never invent numbers.",
    tools=[verdict_tool],                                                   # the only tool Claude may call
    tool_choice={"type": "tool", "name": "submit_ad_verdict"},             # FORCE that tool — no free-text reply
    messages=[{"role": "user", "content": verdict_message}],
)

# Find the tool_use block in the response. content is a LIST of blocks (R1.8); we want the tool_use one.
verdict_dict = None
for block in verdict_response.content:
    # The block we want announces itself with type == "tool_use".
    if block.type == "tool_use":
        # block.input is a plain dict that already conforms to our schema — Anthropic enforced it at generation.
        verdict_dict = block.input
        break

# Wrap the dict back into our Pydantic model for type-safe, dot-access reading (and a final validation).
final_verdict = AdVerdict(**verdict_dict)

# Read the guaranteed-shape fields with a dot, exactly as in R1.13/R1.14.
print("Structured verdict, shape guaranteed by the schema:")
print("  headline          :", final_verdict.headline)
print("  winner            :", final_verdict.winner)
print("  relative_lift_pct :", final_verdict.relative_lift_pct)

Compare this with Pattern 2 and feel the difference. In Pattern 2 we *asked* for JSON and braced ourselves to validate (and possibly retry) afterward. Here, `tool_choice` *forced* Claude to call our tool, and to do so it *had to* produce arguments matching `AdVerdict`'s schema. The structure is guaranteed at the moment of generation; `block.input` arrives already conforming; wrapping it in `AdVerdict(**verdict_dict)` is a final belt-and-braces check that essentially never fails.

This is why Pattern 3 is the one we reach for whenever the output must be **structured and reliable** — a plan with fixed fields, a verdict with set columns, anything that feeds the next step of a pipeline. It is also, as promised in R1.9, the reason this course never needs a second vendor: Anthropic's tool use already delivers generation-time schema enforcement.

> **A note on the `**` in `AdVerdict(**verdict_dict)`.** The `**` "unpacks" a dictionary into keyword arguments. If `verdict_dict` is `{"headline": "...", "winner": "B", "relative_lift_pct": 30.0}`, then `AdVerdict(**verdict_dict)` is shorthand for `AdVerdict(headline="...", winner="B", relative_lift_pct=30.0)`. It is just a tidy way to turn a dict into the arguments a class expects.

Three patterns, one rule. In every one of them, Python computed the numbers and Claude only chose the words. Let us now step back and decide *when to use which*.

***
## R1.16 Which pattern should I reach for?

You now own three tools. Here is the simple decision procedure for choosing among them — keep it nearby until it becomes second nature.

| If you need… | Use | Why |
|---|---|---|
| Prose for a human to read (a memo, a Slack note, a caption) | **Pattern 1 — plain text** (R1.11) | The output *is* the writing; there are no fields to extract. Simplest possible call. |
| A few labelled fields, and a small chance of a malformed reply is acceptable | **Pattern 2 — free-form JSON** (R1.12) | Lightweight structure. You validate with `json.loads` and handle the rare failure yourself. |
| Reliable structured output that the next step of code depends on | **Pattern 3 — tool use + Pydantic** (R1.15) | The schema is enforced *at generation time*; the reply is guaranteed to fit. The robust default for anything feeding a pipeline. |

A flatter way to remember it:

1. **Is the answer meant to be *read by a person*?** → **Pattern 1**. Stop here.
2. **Does my *code* need to pull fields out of the answer?** → you need structure, so go to question 3.
3. **Does it matter if the structure is occasionally wrong?**
   - "Not really, I'll catch it" → **Pattern 2**.
   - "Yes — downstream code depends on it" → **Pattern 3**.

In practice, once you are comfortable, **Pattern 1 for writing and Pattern 3 for structure** will cover almost everything you do as an analyst, with Pattern 2 as the quick-and-light middle option. And underneath all three, unchanged, sits the rule from R1.2: *Python computes, the LLM interprets.*

***
## R1.17 Pitfalls, costs, and best practices

A short field guide to staying safe, cheap, and honest. None of this is advanced; all of it will save you a bad afternoon.

### Money and tokens

- **You pay per token, in both directions.** Recall from R1.8 that each call reports `response.usage.input_tokens` and `output_tokens`. You are billed for *both* the prompt you send and the reply you receive. Long prompts cost money too, not just long answers.
- **`max_tokens` caps the *output*, not the cost of the input.** Setting `max_tokens=200` bounds the reply length; it does nothing about a 5,000-token prompt you sent. Keep prompts lean — send the numbers and the instruction, not your entire dataset.
- **Pick the smallest model that works (R1.9).** Haiku is dramatically cheaper than Opus. For "interpret these numbers," Haiku is the right and frugal choice.
- **The toy calls in this notebook cost a fraction of a cent in total.** But the habit of watching `usage` is worth forming now, before you are looping over thousands of rows.

### Safety and secrets

- **Never put your API key in code, notebooks, or git.** It lives in the `ANTHROPIC_API_KEY` environment variable (R1.5–R1.6), full stop. A key committed to a public repository can be found and abused within minutes.
- **Treat model output as a draft, not gospel.** Even with Pattern 3 guaranteeing *shape*, you the analyst are responsible for the *content*. Read what Claude wrote before it goes to a stakeholder.

### Honesty (the rule that outranks all others)

- **Python computes; the LLM interprets (R1.2).** Never let a number reach a stakeholder that was not computed in Python. Put the figures in the prompt; forbid invention explicitly; check the output.
- **Do not ask the model to do statistics.** Significance tests, confidence intervals, regressions — those are Python's job (Sessions 03–04). The model's job is to phrase the result the analysis already produced.
- **Reproducibility.** Models can word the same request differently on different runs. The *numbers* are fixed because Python produced them; the *prose* may vary. That is fine for a memo, but it is one more reason structured data (Pattern 3) is safer than free prose when exact fields matter.

### Reliability

- **Always check `stop_reason` (R1.8).** A `"max_tokens"` means a truncated answer — raise the cap.
- **Calls can fail.** Networks hiccup; services rate-limit. In real production code you would wrap calls in error handling and retries. For this course's small, occasional calls we keep it simple, but know that robustness is a real concern at scale.

Live by the money/secrets/honesty trio and you will use this tool responsibly for the rest of the course.

***
## R1.18 References — where to go deeper

Everything in this notebook is drawn from, and expanded by, the resources below. They are grouped by topic and every link is live. You do not need all of them — pick what matches the part you want to strengthen.

### Anthropic — the official documentation (your primary source)

- [Intro to Claude](https://platform.claude.com/docs/en/intro) — the friendly front door to the whole developer platform.
- [Messages API reference](https://platform.claude.com/docs/en/api/messages) — the exact specification of `client.messages.create(...)`: every parameter (`model`, `max_tokens`, `system`, `messages`, `tools`, `tool_choice`) and the response shape. Pairs with R1.7–R1.8.
- [Using the Messages API](https://platform.claude.com/docs/en/build-with-claude/working-with-messages) — a worked walkthrough of building the `messages` list, including multi-turn conversations and the assistant-prefill trick from R1.12.
- [Tool use overview](https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview) and [How tool use works](https://platform.claude.com/docs/en/agents-and-tools/tool-use/how-tool-use-works) — the feature behind Pattern 3 (R1.15), explained in full.
- [Models overview](https://platform.claude.com/docs/en/about-claude/models/overview) and [Pricing](https://platform.claude.com/docs/en/about-claude/pricing) — the always-current Haiku / Sonnet / Opus line-up and what each costs. This is why R1.9 hard-codes no prices: check here instead.
- [Anthropic Academy: Build with Claude](https://www.anthropic.com/learn/build-with-claude) — Anthropic's own free, beginner-oriented course on calling the API from scratch. The best first stop if you want a guided tour beyond this notebook.

### Anthropic — prompt engineering (the R1.10 deep-dive)

- [Prompt engineering overview](https://platform.claude.com/docs/en/build-with-claude/prompt-engineering/overview) — the structured guide to the core techniques (be clear and direct, use examples, let the model think, XML tags, role prompting, prompt chaining). Start here.
- [Prompting best practices](https://platform.claude.com/docs/en/build-with-claude/prompt-engineering/claude-prompting-best-practices) — the living, model-specific reference for getting the most out of Claude's latest models.
- [The Interactive Prompt Engineering Tutorial](https://github.com/anthropics/prompt-eng-interactive-tutorial) — a hands-on, nine-chapter GitHub course you run yourself; it uses Haiku, exactly as we do. The single best way to *practise* prompting.
- [Anthropic Cookbooks](https://github.com/anthropics/claude-cookbooks) and the broader [Anthropic Courses](https://github.com/anthropics/courses) — runnable example notebooks for common real-world tasks.

### The Anthropic Python SDK

- [`anthropic-sdk-python` on GitHub](https://github.com/anthropics/anthropic-sdk-python) — the source and README for the library we `import anthropic`. Skim the README's quickstart to see the same call you wrote in R1.7.
- [`anthropic` on PyPI](https://pypi.org/project/anthropic/) — the package page; this is what `pip install anthropic` fetches.

### Python building blocks (R1.3, R1.4, R1.13)

- [Python tutorial: Data structures](https://docs.python.org/3/tutorial/datastructures.html) — the official, authoritative explanation of lists, tuples, and dictionaries. The reference behind R1.3.
- [Python tutorial: Classes](https://docs.python.org/3/tutorial/classes.html) — the official walkthrough of classes, objects, `__init__`, and `self`. The reference behind R1.13.
- [Python `json` module](https://docs.python.org/3/library/json.html) — the documentation for `json.dumps` and `json.loads` from R1.4.
- **Corey Schafer — [Python: Lists, Tuples, and Sets](https://www.youtube.com/watch?v=W8KRzm-HUcc)** (YouTube) — a calm, beginner-perfect video tour of Python's containers. The gentlest possible companion to R1.3.
- **Corey Schafer — [Python OOP Tutorials playlist](https://www.youtube.com/playlist?list=PL-osiE80TeTsqhIuOqKhwlXsIBIdSeYtc)**, starting with [Classes and Instances](https://www.youtube.com/watch?v=ZDa-Z5JzLYM) (YouTube) — the clearest free introduction to classes and objects on the internet. Watch the first video and R1.13 will click permanently.

### Pydantic (R1.14)

- [Pydantic documentation (v2)](https://docs.pydantic.dev/latest/) — the official home, with the [Models](https://docs.pydantic.dev/latest/concepts/models/) and [Fields](https://docs.pydantic.dev/latest/concepts/fields/) concept pages covering `BaseModel`, `Field`, and validation — exactly the R1.14 material.
- **Real Python — [Pydantic: Simplifying Data Validation in Python](https://realpython.com/python-pydantic/)** (article) and its [companion video course](https://realpython.com/videos/understanding-pydantic/) — a thorough, friendly, example-led walkthrough of Pydantic v2 for someone who has just met classes.

### What an API is (R1.5)

- [MDN — Introduction to web APIs](https://developer.mozilla.org/en-US/docs/Learn_web_development/Extensions/Client-side_APIs/Introduction) — a clear, vendor-neutral explanation of what APIs are and how request/response works.
- [MDN — REST glossary entry](https://developer.mozilla.org/en-US/docs/Glossary/REST) — a one-page definition of the REST/HTTP style most web APIs (including Anthropic's) follow.

> **A suggested path after this notebook:** work through **Anthropic's Interactive Prompt Engineering Tutorial** (you now understand every line of its code, so you can focus on the prompting craft), then skim a couple of the **Anthropic Cookbook** notebooks to see the three patterns applied to real tasks. From there, you are ready to use Claude confidently in your own analysis.

This closes the first review. From **Session 05** onward we return to new statistics — but the Claude toolkit you have just consolidated travels with you, unchanged, into every session that follows. You will reach for Pattern 1 to write up findings and Pattern 3 to structure them, and you will never again wonder what is happening inside that little box at the end of the notebook.

<hr>

![](../_img/DK_Logo_White_150.png)

DataKolektiv, 2026.

[hello@datakolektiv.com](mailto:hello@datakolektiv.com)

<font size=1>License: [GPLv3](../LICENSE). This Notebook is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version. This Notebook is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.</font>